# 05 - Trasferimenti INPS e distribuzioni

Questo notebook copre due punti centrali della live:

- trasferimenti dallo Stato a INPS e loro scomposizione;
- distribuzione separata di pensioni, pensionati e reddito pensionistico.

Le celle funzionano anche se le tabelle finali sono ancora vuote.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CODE_DIR = REPO_ROOT / 'code'
DATA_FINAL_DIR = REPO_ROOT / 'data' / 'final'
METADATA_DIR = REPO_ROOT / 'metadata'

if str(CODE_DIR) not in sys.path:
    sys.path.append(str(CODE_DIR))

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)

## Parametri modificabili

In [ ]:
ANNO = None
INDICATORE_TRASFERIMENTI_FINALITA = 'trasferimenti_stato_inps_per_finalita'
INDICATORE_TRASFERIMENTI_VOCE = 'trasferimenti_stato_inps_per_voce'
INDICATORE_PENSIONI_CLASSE = 'pensioni_per_classe_importo'
INDICATORE_PENSIONATI_REDDITO = 'pensionati_per_classe_reddito_pensionistico'
INDICATORE_SPESA_CLASSE = 'spesa_per_classe_importo'

In [ ]:
def read_csv_optional(path: Path) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_csv(path)


def filter_indicator(df: pd.DataFrame, indicatore_id: str) -> pd.DataFrame:
    if df.empty or 'indicatore_id' not in df.columns:
        return pd.DataFrame()
    out = df[df['indicatore_id'].astype(str).eq(indicatore_id)].copy()
    if ANNO is not None and 'anno' in out.columns:
        out = out[pd.to_numeric(out['anno'], errors='coerce').eq(ANNO)]
    return out


def plot_bar(df: pd.DataFrame, category: str, title: str, ylabel: str) -> None:
    if df.empty:
        print('Dati non disponibili:', title)
        return
    if category not in df.columns or 'valore' not in df.columns:
        print('Colonne richieste non disponibili per:', title)
        return
    data = df.copy()
    data['valore'] = pd.to_numeric(data['valore'], errors='coerce')
    data = data.dropna(subset=[category, 'valore']).sort_values('valore', ascending=False)
    if data.empty:
        print('Nessun valore numerico:', title)
        return
    plt.figure(figsize=(10, 6))
    plt.bar(data[category].astype(str), data['valore'])
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## Classificazione dei trasferimenti

Questa tabella descrive come ricondurre le voci ufficiali a categorie analitiche leggibili.

In [ ]:
classificazione = read_csv_optional(METADATA_DIR / 'classificazione_trasferimenti_inps.csv')
display(classificazione)

## Trasferimenti dallo Stato a INPS

La tabella `tabella_trasferimenti_inps.csv` deve contenere le voci di trasferimento, la categoria analitica e la finalita'.

In [ ]:
trasferimenti = read_csv_optional(DATA_FINAL_DIR / 'tabella_trasferimenti_inps.csv')
print('Righe trasferimenti:', len(trasferimenti))
display(trasferimenti.head(30))

In [ ]:
per_finalita = filter_indicator(trasferimenti, INDICATORE_TRASFERIMENTI_FINALITA)
plot_bar(per_finalita, 'categoria_analitica', 'Trasferimenti Stato-INPS per categoria', 'Euro')

per_voce = filter_indicator(trasferimenti, INDICATORE_TRASFERIMENTI_VOCE)
plot_bar(per_voce.head(20), 'voce_nome', 'Prime voci dei trasferimenti Stato-INPS', 'Euro')

## Distribuzione di pensioni e pensionati

Questa parte tiene separate tre cose:

- numero di pensioni per classe di importo;
- numero di pensionati per classe di reddito pensionistico;
- spesa o reddito pensionistico per classe.

In [ ]:
distribuzione = read_csv_optional(DATA_FINAL_DIR / 'tabella_distribuzione_pensionati.csv')
print('Righe distribuzione:', len(distribuzione))
display(distribuzione.head(30))

In [ ]:
pensioni_classe = filter_indicator(distribuzione, INDICATORE_PENSIONI_CLASSE)
pensionati_reddito = filter_indicator(distribuzione, INDICATORE_PENSIONATI_REDDITO)
spesa_classe = filter_indicator(distribuzione, INDICATORE_SPESA_CLASSE)

plot_bar(pensioni_classe, 'classe_importo', 'Pensioni per classe di importo', 'Numero')
plot_bar(pensionati_reddito, 'classe_importo', 'Pensionati per classe di reddito pensionistico', 'Numero')
plot_bar(spesa_classe, 'classe_importo', 'Spesa per classe di importo', 'Euro')

## Lettura corretta

La live insiste su un punto: una pensione non coincide con un pensionato. Per valutare la condizione economica delle persone serve il reddito pensionistico complessivo del beneficiario, non solo l importo della singola prestazione.